# Lab 1: Build the Multi-Agent System

Insurance claims involve document review, policy lookup, fraud detection, and summarization — too much for a single prompt to handle well. In this lab, you'll build four specialized agents that collaborate through a graph workflow to process claims end-to-end.

**Why split the work across agents?**
- Each agent gets a focused prompt optimized for one task
- When something breaks, you know exactly which agent failed
- Independent steps (policy retrieval and inspection) run in parallel, reducing latency
- Conditional routing means downstream agents don't execute if upstream validation fails — saving tokens and compute

**What you'll build:**

| Agent | Responsibility |
|-------|---------------|
| Document Analysis | Extracts structured claim data from text and images |
| Policy Retrieval | Looks up relevant coverage and deductibles |
| Inspection | Identifies fraud indicators and risk factors |
| Claim Summary | Compiles everything into an actionable report |

**Prerequisites:** AWS account with Bedrock access (Claude Haiku enabled)

**Time:** 30–45 minutes

## Step 1: Prerequisites and Setup

Install the Strands Agents SDK (multi-agent framework), Pydantic (typed data models), boto3 (Bedrock client), and the OpenTelemetry API (instrumentation used in later labs). AWS credentials should already be configured on your SageMaker notebook via the instance role — no manual setup needed.


In [1]:
# Install workshop dependencies. These pins are the versions the workshop was tested on.
!pip install --quiet \
    'strands-agents[otel]==1.37.0' \
    'bedrock-agentcore-starter-toolkit>=0.1,<1.0' \
    'aws-opentelemetry-distro>=0.10,<1.0' \
    'pydantic>=2.10,<3.0' \
    'boto3>=1.35,<2.0' \
    'awscli>=1.35,<2.0' \
    'opentelemetry-api>=1.30,<2.0'



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Import necessary libraries
import os
from typing import List

from pydantic import BaseModel, Field
from strands import Agent
from strands.multiagent import GraphBuilder
from strands.multiagent.base import MultiAgentResult, Status
from strands.multiagent.graph import GraphState
from strands.models import BedrockModel
from strands.types.content import ContentBlock


## Step 2: Data Models and Configuration

Set up three things before building the agents:

1. **Pydantic data models** — type-safe structures for claims and analysis outputs. Critical when one agent's output becomes another agent's input.
2. **Bedrock model configuration** — which Claude model and region your agents will call.
3. **A sample claim** — test data to run through the pipeline at the end.


In [ ]:
# Define core data models
class ClaimData(BaseModel):
    """Insurance claim data structure"""
    claim_id: str
    policy_number: str
    claimant_name: str
    incident_date: str
    incident_type: str
    description: str
    estimated_damage: float
    documents: List[str]  # File paths
    location: str

class ProcessingResult(BaseModel):
    """Analysis result structure"""
    claim_id: str
    status: str
    analysis_notes: List[str] = []

# Structured output models for document analysis
class AnalyzedDocument(BaseModel):
    content: str = Field(description="Extracted text content")
    extension: str = Field(description="Document type")

class AnalyzedImages(BaseModel):
    description: str = Field(description="Image analysis")
    file_name: str = Field(description="File name")
    extension: str = Field(description="Image type")

class DocumentAnalysisResult(BaseModel):
    analyzed_documents: List[AnalyzedDocument] = Field(description="Document analysis")
    analyzed_images: List[AnalyzedImages] = Field(description="Image analysis")
    status: str = Field(description="Analysis status")
    summary: str = Field(description="Analysis summary")

In [ ]:
# Configuration setup
def get_bedrock_model():
    """Configure Amazon Bedrock model"""
    return BedrockModel(
        model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        region_name="us-east-1",
        temperature=0.1
    )

# Initialize model
model = get_bedrock_model()
print("Model configured successfully")

In [ ]:
# Sample claim data preparation
sample_claim = ClaimData(
    claim_id="CLM-2024-001",
    policy_number="POL-AUTO-12345",
    claimant_name="John Smith",
    incident_date="2024-01-15",
    incident_type="Auto Accident",
    description="Rear-end collision with moderate damage to vehicle",
    estimated_damage=8500.0,
    documents=[],  # Add image/document paths here to analyze real claim evidence
    location="Main St & Oak Ave, Orlando, Florida"
)

print(f"Sample claim created: {sample_claim.claim_id}")

## Step 3: Build the Individual Agents

Each agent gets a focused system prompt defining its single responsibility. The four specialists are defined in the order they'll execute: document analysis first, then policy retrieval and inspection in parallel, then claim summary to combine the results.
Pay attention to system prompts for each one


In [ ]:
# Document Analysis Agent
def document_analysis_agent():
    """Agent for analyzing documents and images with structured output"""
    return Agent(
        name="DocumentAnalysisAgent",
        model=model,
        system_prompt="""
        You are a document analysis specialist for insurance claims.
        Analyze provided documents and images to extract relevant information.
        Return structured JSON output with analysis results.
        Set status to 'valid' if analysis is successful, 'invalid' if documents are insufficient.
        """
    )

print("Document Analysis Agent created")

In [ ]:
# Policy Retrieval Agent
def policy_retrieval_agent():
    """Agent for retrieving and analyzing policy information"""
    return Agent(
        name="PolicyRetrievalAgent",
        model=model,
        system_prompt="""
        You are a policy analysis specialist.
        Analyze insurance policies to determine coverage, deductibles, and applicable terms.
        Provide clear coverage analysis for the claim.
        """
    )

print("Policy Retrieval Agent created")

In [ ]:
# Inspection Agent
def inspection_agent():
    """Agent for identifying inconsistencies and potential issues"""
    return Agent(
        name="InspectionAgent",
        model=model,
        system_prompt="""
        You are an inspection specialist for insurance claims.
        Analyze claims for inconsistencies, unusual patterns, or areas requiring attention.
        Highlight any discrepancies or points that need human expert review.
        """
    )

print("Inspection Agent created")

In [ ]:
# Claim Summary Agent
def claim_summary_agent():
    """Agent for compiling comprehensive claim analysis"""
    return Agent(
        name="ClaimSummaryAgent",
        model=model,
        system_prompt="""
        You are a claims summary specialist.
        Compile comprehensive analysis from all previous agents into a clear summary.
        Provide actionable insights for human insurance experts.
        Include key findings, coverage details, and recommendations.
        """
    )

print("Claim Summary Agent created")

## Step 4: Wire Up the Graph

Now connect the agents into a directed graph with conditional routing. The key design decisions:

- Document Analysis is the entry point — everything depends on it
- Policy Retrieval and Inspection run **in parallel** (they don't depend on each other)
- Claim Summary only runs after all three upstream agents complete
- If Document Analysis returns `invalid`, downstream agents are skipped entirely

```
Document Analysis → [Parallel] → Policy Retrieval
                               → Inspection
                 → [Join]      → Claim Summary
```

In [ ]:
# Initialize GraphBuilder
def create_analysis_graph():
    """Build the insurance claims analysis graph"""
    builder = GraphBuilder()
    return builder

# Create graph builder instance
graph_builder = create_analysis_graph()
print("Graph builder initialized")

In [ ]:
# Add agent nodes to the graph
graph_builder.add_node(document_analysis_agent(), "document_analysis")
graph_builder.add_node(policy_retrieval_agent(), "policy_retrieval")
graph_builder.add_node(inspection_agent(), "inspection")
graph_builder.add_node(claim_summary_agent(), "claim_summary")

print("All agent nodes added to graph")

In [ ]:
# Define conditional edge functions
def document_analysis_valid(state: GraphState) -> bool:
    """Check if document analysis completed successfully.

    If structured output is available and exposes a ``status`` field,
    require it to be 'valid'. Otherwise treat node completion as valid.
    """
    node_result = state.results.get("document_analysis")
    if not node_result or node_result.status != Status.COMPLETED:
        return False
    structured = getattr(node_result.result, "structured_output", None)
    if structured is not None and hasattr(structured, "status"):
        return structured.status == "valid"
    return True

def all_dependencies_complete(required_nodes: List[str]):
    """Factory function to create AND condition for multiple dependencies"""
    def check_all_complete(state: GraphState) -> bool:
        return all(
            node_id in state.results and state.results[node_id].status == Status.COMPLETED
            for node_id in required_nodes
        )
    return check_all_complete

print("Conditional functions defined")

In [ ]:
# Define edges with conditional routing
# Document analysis is entry point, others depend on valid analysis
graph_builder.add_edge(
    "document_analysis", "policy_retrieval", 
    condition=document_analysis_valid
)
graph_builder.add_edge(
    "document_analysis", "inspection", 
    condition=document_analysis_valid
)

# Claim summary depends on all other agents completing
dependency_condition = all_dependencies_complete(
    ["document_analysis", "policy_retrieval", "inspection"]
)

graph_builder.add_edge("policy_retrieval", "claim_summary", condition=dependency_condition)
graph_builder.add_edge("inspection", "claim_summary", condition=dependency_condition)

print("Graph edges defined with conditional routing")

In [ ]:
# Set execution parameters
graph_builder.set_entry_point("document_analysis")
graph_builder.set_execution_timeout(900)  # 15 minutes
graph_builder.set_node_timeout(300)       # 5 minutes per node

# Build the final graph
analysis_graph = graph_builder.build()
print("Analysis graph built successfully")

## Step 5: Prepare Multimodal Content

The Document Analysis agent needs to handle both text descriptions and images (e.g., photos of vehicle damage). This function packages claim data into content blocks that Bedrock can process.

In [ ]:
# Multimodal content preparation
def prepare_multimodal_content(claim_data: ClaimData) -> List[ContentBlock]:
    """Prepare content blocks for analysis"""
    content_blocks = []
    
    # Add claim description
    if claim_data.description:
        content_blocks.append({"text": f"Claim Description: {claim_data.description}"})
    
    # Add instruction
    content_blocks.append({
        "text": "Analyze the following files and extract structured information:"
    })
    
    # Process files
    for file_path in claim_data.documents:
        if not os.path.exists(file_path):
            continue
            
        file_ext = os.path.splitext(file_path)[1].lower()
        
        if file_ext in [".png", ".jpg", ".jpeg"]:
            with open(file_path, "rb") as f:
                image_bytes = f.read()
            content_blocks.append({
                "image": {
                    "format": file_ext[1:],
                    "source": {"bytes": image_bytes}
                }
            })
    
    return content_blocks

print("Content preparation function defined")

In [ ]:
# Context preparation
def prepare_claim_context(claim_data: ClaimData) -> str:
    """Prepare comprehensive context for analysis"""
    return f"""
    CLAIM INFORMATION:
    - Claim ID: {claim_data.claim_id}
    - Policy Number: {claim_data.policy_number}
    - Claimant: {claim_data.claimant_name}
    - Incident Date: {claim_data.incident_date}
    - Incident Type: {claim_data.incident_type}
    - Location: {claim_data.location}
    - Description: {claim_data.description}
    - Estimated Damage: ${claim_data.estimated_damage:,.2f}
    - Supporting Evidence: {', '.join(claim_data.documents)}
    """

print("Context preparation function defined")

## Step 6: Orchestrate the Full Pipeline

The `ClaimsAssistant` class ties everything together — it takes a claim, prepares the content, and runs it through the graph.

In [ ]:
# Main ClaimsAssistant class
class ClaimsAssistant:
    """Main insurance claims analysis system"""
    
    def __init__(self):
        self.graph = analysis_graph
    
    def analyze_claim(self, claim_data: ClaimData) -> MultiAgentResult:
        """Analyze insurance claim through multi-agent graph"""
        print(f"Starting analysis for claim {claim_data.claim_id}")
        
        # Prepare context and content
        claim_context = prepare_claim_context(claim_data)
        multimodal_blocks = prepare_multimodal_content(claim_data)
        
        # Enhanced context with instructions
        enhanced_context = f"""
        You are an insurance claims assistant coordinating specialized agents.
        Analyze the claim comprehensively for human expert review.
        
        Claim Context:
        {claim_context}
        """
        
        # Combine content
        content_blocks = [ContentBlock(text=enhanced_context)] + multimodal_blocks
        
        # Execute graph (let exceptions surface for easier debugging)
        result = self.graph(content_blocks)
        print(f"Analysis completed: {result.status}")
        return result

# Create assistant instance
claims_assistant = ClaimsAssistant()
print("Claims Assistant initialized")

## Step 7: Test the System

Run a test claim through the full pipeline and inspect the results from each agent.

In [ ]:
# Sample claim creation
test_claim = ClaimData(
    claim_id="CLM-TEST-001",
    policy_number="POL-AUTO-12345",
    claimant_name="Jane Doe",
    incident_date="2024-01-20",
    incident_type="Vehicle Collision",
    description="Minor fender bender in parking lot with scratches on rear bumper",
    estimated_damage=2500.0,
    documents=[],  # Add image/document paths here to analyze real claim evidence
    location="Shopping Center Parking Lot, Miami, FL"
)

print(f"Test claim created: {test_claim.claim_id}")
print(f"Incident: {test_claim.incident_type}")
print(f"Estimated damage: ${test_claim.estimated_damage:,.2f}")

In [ ]:
# Execute analysis
print("Executing multi-agent analysis...")
print("=" * 50)

# Run the analysis
analysis_result = claims_assistant.analyze_claim(test_claim)

print("\nAnalysis Results:")
print("=" * 50)
print(f"Status: {analysis_result.status}")
completed = getattr(analysis_result, "completed_nodes", None)
total = getattr(analysis_result, "total_nodes", None)
if completed is not None and total is not None:
    print(f"Completed nodes: {completed}/{total}")
exec_time = getattr(analysis_result, "execution_time", None)
if exec_time is not None:
    print(f"Execution time: {exec_time}ms")

# Display individual agent results
print("\nAgent Results:")
for node_id, node_result in getattr(analysis_result, "results", {}).items():
    print(f"\n{node_id.upper()}:")
    print("-" * 30)
    if node_result.result and hasattr(node_result.result, 'message'):
        message = node_result.result.message
        if hasattr(message, 'content') and message.content:
            content = str(message.content[0].text)[:200] + "..."
            print(content)
        else:
            print(str(message)[:200] + "...")
    else:
        print("No result available")

print("\nAnalysis complete!")

## What You've Built

You now have a working multi-agent insurance claims pipeline:

| Component | What It Does |
|-----------|-------------|
| 4 specialized agents | Each with a focused prompt and clear responsibility |
| Graph orchestration | Conditional routing and parallel execution via GraphBuilder |
| Structured outputs | Pydantic models ensuring type-safe data flow between agents |
| Multimodal support | Text and image processing in a single pipeline |

**What you should verify before moving on:**
- The graph executed all four agents successfully
- Document Analysis returned a `valid` status
- Policy Retrieval and Inspection ran in parallel
- Claim Summary produced a human-readable report

**What you can't answer yet:** How much did that claim cost to process? Which agent was the bottleneck? Are the results actually correct? That's what Lab 2 (operational metrics) and Lab 3 (quality evaluation) are for.

Continue to **Lab 2** to instrument this pipeline with CloudWatch metrics.